# ODE AI — Colab runner (T4)

Config lives in the next cell. Re-run training after a disconnect; `resume=true` loads `last.pt` from Drive.

In [ ]:
# === Config (edit these) ===
REPO_URL = "https://github.com/YOUR_USER/YOUR_REPO.git"  # set me
BRANCH = "main"
DRIVE_ROOT = "/content/drive/MyDrive/ode_ai"
GEN_MAX_SECONDS = 600
EPOCHS = 20
BATCH_SIZE = 64
EVAL_MAX_EXAMPLES = 200


## 1. GPU check, Drive mount, folders

In [ ]:
!nvidia-smi
from google.colab import drive
drive.mount("/content/drive")
from pathlib import Path
Path(DRIVE_ROOT).mkdir(parents=True, exist_ok=True)
Path(f"{DRIVE_ROOT}/data").mkdir(parents=True, exist_ok=True)
Path(f"{DRIVE_ROOT}/checkpoints").mkdir(parents=True, exist_ok=True)
print("DRIVE_ROOT=", DRIVE_ROOT)


## 2. Clone / pull repo (optional private token)

In [ ]:
import os
import subprocess
from pathlib import Path

def _clone_url(repo_url: str) -> str:
    try:
        from google.colab import userdata
        token = userdata.get("GITHUB_TOKEN")
    except Exception:
        token = None
    if not token:
        return repo_url
    if repo_url.startswith("https://"):
        return "https://" + token + "@" + repo_url[len("https://"):]
    return repo_url

repo_dir = Path("/content/ode_ai_repo")
url = _clone_url(REPO_URL)
if (repo_dir / ".git").exists():
    subprocess.run(["git", "fetch", "origin"], cwd=repo_dir, check=True)
    subprocess.run(["git", "checkout", BRANCH], cwd=repo_dir, check=True)
    subprocess.run(["git", "pull", "origin", BRANCH], cwd=repo_dir, check=True)
else:
    subprocess.run(
        ["git", "clone", "--branch", BRANCH, url, str(repo_dir)],
        check=True,
    )
os.chdir(repo_dir)
print("cwd=", Path.cwd())


## 3. Install requirements

In [ ]:
!pip install -q -r requirements.txt
import sympy, torch
print("sympy=", sympy.__version__)
print("torch=", torch.__version__)
print("cuda_available=", torch.cuda.is_available())


## 4. Data: restore from Drive or generate, then finalize

In [ ]:
import json
import shutil
import subprocess
from pathlib import Path

drive_all = Path(DRIVE_ROOT) / "data" / "all.jsonl"
local_all = Path("data/processed/all.jsonl")
local_all.parent.mkdir(parents=True, exist_ok=True)

if drive_all.exists():
    shutil.copy2(drive_all, local_all)
    print("Restored", drive_all, "->", local_all)
else:
    subprocess.run(
        [
            "python",
            "scripts/generate_dataset.py",
            "--config",
            "configs/default.yaml",
            "--max-seconds",
            str(GEN_MAX_SECONDS),
        ],
        check=True,
    )
    if local_all.exists():
        shutil.copy2(local_all, drive_all)
        print("Saved", local_all, "->", drive_all)

subprocess.run(
    ["python", "scripts/generate_dataset.py", "--config", "configs/default.yaml", "--finalize"],
    check=True,
)
meta = json.loads(Path("data/processed/meta.json").read_text())
print("split sizes:", {k: meta[k] for k in ("n_train", "n_val", "n_test", "n_test_unseen")})
print("versions:", meta.get("sympy_version"), meta.get("torch_version"))


## 5. Write configs/colab.yaml

In [ ]:
from pathlib import Path
import yaml

cfg = yaml.safe_load(Path("configs/default.yaml").read_text())
cfg["train"]["checkpoint_dir"] = f"{DRIVE_ROOT}/checkpoints"
cfg["train"]["epochs"] = EPOCHS
cfg["train"]["batch_size"] = BATCH_SIZE
cfg["train"]["resume"] = True
Path("configs/colab.yaml").write_text(yaml.safe_dump(cfg, sort_keys=False))
print(Path("configs/colab.yaml").read_text())


## 6. Speed test (~5 min)

In [ ]:
import time
import subprocess
from pathlib import Path

t0 = time.perf_counter()
subprocess.run(
    ["python", "-m", "src.train", "--config", "configs/colab.yaml", "--max-seconds", "300"],
    check=True,
)
elapsed = time.perf_counter() - t0
metrics = Path(f"{DRIVE_ROOT}/checkpoints/metrics.jsonl")
epochs_done = 0
if metrics.exists():
    epochs_done = sum(1 for _ in metrics.open())
print(f"speed_test_elapsed_s={elapsed:.1f} metrics_epochs={epochs_done}")
if epochs_done > 0:
    print(f"est_seconds_per_epoch={elapsed / epochs_done:.1f}")
else:
    print("est_seconds_per_epoch=unknown (no completed epoch in metrics)")


## 7. Full training

**Note:** Re-running this cell resumes from `last.pt` in `DRIVE_ROOT/checkpoints` after a Colab disconnect (`resume=true`).

In [ ]:
!python -m src.train --config configs/colab.yaml


## 8. Evaluation (val / test / test_unseen)

In [ ]:
import subprocess

ckpt = f"{DRIVE_ROOT}/checkpoints/best.pt"
for split in ("val", "test", "test_unseen"):
    print("=" * 60, split)
    subprocess.run(
        [
            "python",
            "-m",
            "src.evaluate",
            "--config",
            "configs/colab.yaml",
            "--checkpoint",
            ckpt,
            "--split",
            split,
            "--max-examples",
            str(EVAL_MAX_EXAMPLES),
        ],
        check=True,
    )


## 9. Demo: solve()

In [ ]:
from src.solve import load_model_and_tokenizer, solve

model, tokenizer, cfg, device = load_model_and_tokenizer(f"{DRIVE_ROOT}/checkpoints/best.pt")
examples = [
    "y' = x",
    "y' = 2*x",
    "y' = exp(x)",
    "y' = x*exp(y)",
    "y' = sin(x)",          # outside training distribution
    "y' = y**2 + x",        # outside training distribution
]
for ode in examples:
    r = solve(ode, model, tokenizer, device=device)
    print(ode, "->", "VERIFIED", r["solution"] if r["verified"] else (r.get("error") or "no verified candidate"))


## 10. Plots from metrics.jsonl

In [ ]:
import json
from pathlib import Path
import matplotlib.pyplot as plt

rows = []
path = Path(f"{DRIVE_ROOT}/checkpoints/metrics.jsonl")
with path.open() as fh:
    for line in fh:
        if line.strip():
            rows.append(json.loads(line))
epochs = [r["epoch"] for r in rows]
train_loss = [r.get("train_loss") for r in rows]
val_loss = [r.get("val_loss") for r in rows]
exact = [r.get("exact_match") for r in rows]
fig, ax = plt.subplots(1, 2, figsize=(10, 4))
ax[0].plot(epochs, train_loss, marker="o", label="train_loss")
ax[0].plot(epochs, val_loss, marker="o", label="val_loss")
ax[0].set_title("train/val loss")
ax[0].set_xlabel("epoch")
ax[0].legend()
ax[1].plot(epochs, exact, marker="o")
ax[1].set_title("exact_match")
ax[1].set_xlabel("epoch")
plt.tight_layout()
plt.show()


## Examen

Pipeline del examen final (15 EDO): verificación por diferenciación, red neuronal pequeña por ejercicio, y columna opcional del Transformer (`src.solve.solve`).

- Respuestas del equipo: `exam/answers.py` (dejar `None` = PENDIENTE).
- Demo de prueba: `use_demo=True` (NO son las respuestas del equipo).
- Salidas (CSV/PNG) en Drive para sobrevivir desconexiones.


In [ ]:
# Examen: cargar modelo si aún no existe; correr tabla + figuras
from pathlib import Path

try:
    model
    tokenizer
    device
    print("Reusing model/tokenizer/device already in memory")
except NameError:
    from src.solve import load_model_and_tokenizer
    ckpt = Path(DRIVE_ROOT) / "checkpoints" / "best.pt"
    if not ckpt.exists():
        ckpt = Path(DRIVE_ROOT) / "checkpoints" / "last.pt"
    model, tokenizer, _cfg, device = load_model_and_tokenizer(ckpt)
    print("Loaded", ckpt)

EXAM_OUT = Path(DRIVE_ROOT) / "exam_output"
EXAM_OUT.mkdir(parents=True, exist_ok=True)

from exam.report import run, show_equations
from IPython.display import display, Image

# use_demo=False -> exam/answers.py (equipo). True solo para probar el pipeline.
df_exam = run(
    use_demo=False,
    model=model,
    tokenizer=tokenizer,
    device=device,
    output_dir=EXAM_OUT,
)
display(df_exam)
show_equations(model=model, tokenizer=tokenizer, device=device)
display(Image(filename=str(EXAM_OUT / "exam_nn_error.png")))
display(Image(filename=str(EXAM_OUT / "exam_curves.png")))
print("Saved under", EXAM_OUT)
